In [ ]:
import glob
import numpy as np
import pandas as pd
import datetime as dt

import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

from JunoPy.jedi import JediData, merge_data_species
from JunoPy.index import IndexFile
from JunoPy.mag import MagData
from JunoPy.library import find_ticks_between

In [ ]:
startevent = dt.datetime.strptime('2017-236T20:21:15', '%Y-%jT%H:%M:%S')
endevent   = dt.datetime.strptime('2017-236T20:21:38', '%Y-%jT%H:%M:%S')

# startevent = dt.datetime.strptime('2017-236T00:21:15', '%Y-%jT%H:%M:%S')
# endevent   = dt.datetime.strptime('2017-236T23:21:38', '%Y-%jT%H:%M:%S')
start = startevent - dt.timedelta(minutes=20)
end   = endevent + dt.timedelta(minutes=20)

magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
magdata = MagData(filename=magfile)
magdata.df = magdata.df.loc[start:end]

jedifiles = glob.glob('data/jedi/*_CDR_{:s}_*.TAB'.format(start.strftime('%Y%j')))
if not jedifiles:
    jediindex = IndexFile(dataset='jedi')
    jediindex.get_files_between_dates(
        start, end, DoDownload=True, sid='ERSISP')
    jediindex.get_files_between_dates(
        start, end, DoDownload=True, sid='ERSESP')
    
jedifiles = glob.glob('data/jedi/*_CDR_{:s}_*.TAB'.format(start.strftime('%Y%j')))
jedidatas = []
for jedifile in jedifiles:
    jedidatas.append(JediData(filename=jedifile))
    print(jedifile)
    
ion_data = []
electron_data = []
for jdata in jedidatas:
    
    jdata.df = jdata.df.loc[start:end]
    jdata.calc_broadband_flux()

    if jdata.metadata['particletype'].str.contains('ELECTRON').any():
        electron_data.append(jdata)
    else:
        ion_data.append(jdata)

ion_data = merge_data_species(ion_data, species='ion')
electron_data = merge_data_species(electron_data, species='electron')

In [ ]:
def plot_data(magdata, ion_data, electron_data, start, end, outfolder=None):
    """Plot data between start and end."""
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(4 + 4, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(7, 8), 
                            dpi=150)
    
    axs[:4] = magdata.plot(vars=['Br', 'Btheta', 'Bphi', 'Bmag'], axs=axs[:4])
    
    ion_data.plot_flux_spectra(ax=axs[4])
    electron_data.plot_flux_spectra(ax=axs[5])
    xticks = ion_data.plot_pitchangle_spectra(ax=axs[6], vmin=1e5, vmax=1e7, cmap='jet')
    electron_data.plot_pitchangle_spectra(ax=axs[7], vmin=1e5, vmax=1e7, cmap='jet')
        
        
    
    for a in axs[:-2]:
        a.set_xlim([timearray[0], timearray[-1]])
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.label_outer()
        
    for a in axs[-2:]:
        a.set_xlim([xticks[0], xticks[-1]])
        a.set_xticks(xticks)
        a.set_xticklabels(tickstring)
        a.label_outer()
        
#         a.axvline(dt.datetime.strptime('2018-085T15:53:45', '%Y-%jT%H:%M:%S'), color='red', lw=0.5)
#         a.axvline(dt.datetime.strptime('2018-085T15:54:15', '%Y-%jT%H:%M:%S'), color='red', lw=0.5)
        
#     for a in axs[:4]:
#         a.axvspan(start + dt.timedelta(minutes=5), 
#                   end - dt.timedelta(minutes=5), alpha=0.2, color=[0.2, 0.2, 0.2])
            
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.92)
    
    if outfolder:
        fig.savefig(outfolder + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
        plt.close(fig)
    
    # fig.savefig('out_mva_temp/' + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
    
    # plt.close(fig)
    
plot_data(magdata, ion_data, electron_data, start, end)